In [1]:
""" Used when checked out from Git """
import sys
import os
def add_local():
    d = os.getcwd()
    assert d[-9:] == "notebooks", "This code assume that the notebook is run out of 'notebooks'. Remove this code if not needed"
    sys.path.insert( 0, d[:-10] )
    print(f"Added '{d[:-10]}' to the import path")
add_local()


Added 'C:\Users\hans\OneDrive\Python3\packages\cdxcore' to the import path


In [2]:
from cdxcore.config import Config, Int, Float
from dataclasses import dataclass

@dataclass
class Data:
    data : Config = Config().as_field()

    def f(self):
        return self.data("x", 1, Int>0, "A positive integer")

d = Data()   # default constructor used.
d.f()


1

In [3]:
config = Config()
config = Config()
config['features']           = [ 'time', 'spot' ]   # examplearray-type assignment
config.scaling               = [ 1., 1000. ]        # example object-type assignment
        
config.network.depth         = 10
config.network.activation    = 'relu'
config.network.width         = 100

config.network               = Config()
config.network.depth         = 10
config.network.activation    = 'relu'
config.network.width         = 100

import numpy as np

class Network(object):
    def __init__( self, config ):
        self.depth      = config("depth", 1, Int>0, "Depth of the network")
        self.width      = config("width", 1, Int>0, "Width of the network")
        self.activation = config("activation", "selu", str, "Activation function")
        config.done() # see below

class Model(object):
    def __init__( self, config ):
        # read top level parameters
        self.features = config("features", [], list, "Features for the agent" )
        self.scaling  = config("scaling", [], np.asarray, "Weigths for the agent", help_default="no initial weights")
        self.networks = Network(config.network)
        config.done() # see below
        
model = Model( config )
print("/done")


/done


In [4]:
config = Config()
config = Config()
config['features']           = [ 'time', 'spot' ]   # examplearray-type assignment
config.scaling               = [ 1., 1000. ]        # example object-type assignment
        
class Model(object):
    def __init__( self, config ):
        # read top level parameters
        self.features = config("features", [], list, "Features for the agent" )
        self.scaling  = config("scaling", [], np.asarray, "Weigths for the agent", help_default="no initial weights")
        
model = Model( config )
print( config.usage_report() )
print( config.recorder )

config['features'] = ['time', 'spot'] # Features for the agent; default: []
config['scaling'] = [   1. 1000.] # Weigths for the agent; default: no initial weights

SortedDict({"config['features']": SortedDict({'default': [], 'help': 'Features for the agent', 'help_cast': 'list', 'help_default': '[]', 'raw_use': False, 'value': ['time', 'spot']}), "config['scaling']": SortedDict({'default': [], 'help': 'Weigths for the agent', 'help_cast': 'asarray', 'help_default': 'no initial weights', 'raw_use': False, 'value': array([   1., 1000.])})})


In [9]:
from cdxcore.config import NotDoneError

class Network(object):
    def __init__( self, config ):
        # read top level parameters
        self.depth     = config("depth", 1, Int>=1, "Depth of the network")
        self.width     = config("width", 3, Int>=1, "Width of the network")
        self.activaton = config("activation", "relu", help="Activation function", help_cast="String with the function name, or function")

config                       = Config()
config.features              = ['time', 'spot']
config.network.depth         = 10
config.network.activation    = 'relu'
config.network.widht         = 100   # (intentional typo)

n = Network(config.network)
test_features = config("features", [], list, "Features for my network")

# --> should trigger an error !
try:
    config.done()
except NotDoneError as e:
    print(e)
    


Error closing Config 'config.network': the following config arguments were not read: widht

Summary of all variables read from this object:
config.network['activation'] = relu # Activation function; default: relu
config.network['depth'] = 10 # Depth of the network; default: 1
config.network['width'] = 3 # Width of the network; default: 3
# 
config['features'] = ['time', 'spot'] # Features for my network; default: []



In [11]:


def big_function( cache_dir : str, config : Config = None, **kwargs ):
    assert not cache_dir[-1] in ["/","\\"], cache_dir
    config = Config.config_kwargs( config, kwargs )
    uid    = config.unique_hash(length=8)
    cfile  = f"{cache_dir}/{uid}.pck"

    # attempt to read cache
    try:
        with open(cfile, "rb") as f:
            return pickle.load(f)
    except FileNotFoundError:
        pass

    # do something
    result = config("a", 0, int, "Dummy config") * 1000

    # attempt to read cache
    with open(cfile, "wb") as f:
        pickle.dump(result,f)

    return result

from cdxcore.config import Config
import tempfile as tempfile
import pickle as pickle
import shutil as shutil

tmp_dir  = tempfile.mkdtemp()    
try:
    r = big_function( cache_dir = tmp_dir, a=1 )
finally:
    shutil.rmtree(tmp_dir) # delete test directory
print(r)


1000


In [12]:
from cdxcore.config import Config, Int, Float

def big_function( config ):
    _ = config("activation", "relu", str, "Activation function")
    config.done()

config = Config()
config.no_idea.x = 1
no_idea = config.no_idea.detach()
_ = no_idea("x", 0.)
big_function( config )
print( config.unique_hash(length=8) )
print( config.usage_value_dict() )

config = Config(activation="relu")
big_function( config )
print( config.unique_hash(length=8) )
print( config.usage_value_dict() )


7852e47c
SortedDict({"config.no_idea['x']": 1, "config['activation']": 'relu'})
d715e29c
SortedDict({"config['activation']": 'relu'})


In [14]:
from cdxcore.uniquehash import unique_hash32 as unique_hash, DebugTraceVerbose, Context
import numpy as np

class A(object):
    """ No ID. Because members are protected by default this object is hashed as "empty" """
    def __init__(self, seed = 12312, size = (10,) ):
        np.random.seed( seed )
        self._seed = seed
        self._size = size
        self.__data = np.random.normal( size=size )  # we do not want to hash this: it is determined by the other two parameters
    @property
    def data(self):
        return self.__data

unique_hash( A(), debug_trace=DebugTraceVerbose() )


00: tuple: '(<__main__.A object at 0x0000021B21CCC590>,)'
01:   object with __dict__ A: '<__main__.A object at 0x0000021B21CCC590>'
02:     str: 'A'


'42227e160072fe2830817ccf773fa9a5'